# Chilli Leaf Disease Detector

Classify a chilli leaf image with the trained EfficientNet model. The notebook uses `efficientNet_model.keras` from this folder when present; otherwise it downloads the public model artifact from the Hugging Face Space.

This is an experimental screening demo, not a confirmed diagnosis or pesticide recommendation.

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image
from huggingface_hub import hf_hub_download

MODEL_REPO = "NazmulNIR/chilli-leaf-disease-detector"
MODEL_FILE = "efficientNet_model.keras"
local_model = Path(MODEL_FILE)
if local_model.exists():
    model_path = str(local_model)
else:
    model_path = hf_hub_download(
        repo_id=MODEL_REPO,
        repo_type="space",
        filename=MODEL_FILE,
    )

model = tf.keras.models.load_model(model_path, compile=False)
print(f"Model loaded from: {model_path}")

## Run one prediction

Set `IMAGE_PATH` to a chilli leaf image file available to this notebook.

In [ ]:
CLASS_NAMES = [
    "Chilli __Whitefly",
    "Chilli __Yellowish",
    "Chilli__Anthracnos",
    "Chilli__Damping_Off",
    "Chilli__Leaf_Curl_Virus",
    "Chilli__Leaf_Spot",
    "Chilli__Veinal_Mottle_Virus",
    "Chilli___healthy",
]

def predict_image(image):
    image = image.convert("RGB").resize((224, 224))
    pixels = tf.keras.utils.img_to_array(image)
    scores = model.predict(np.expand_dims(pixels, axis=0), verbose=0)[0]
    return {CLASS_NAMES[i]: float(scores[i]) for i in range(len(CLASS_NAMES))}

IMAGE_PATH = Path("sample_leaf.jpg")  # Change this to your image path
if IMAGE_PATH.exists():
    with Image.open(IMAGE_PATH) as leaf_image:
        prediction_scores = predict_image(leaf_image)
    print(sorted(prediction_scores.items(), key=lambda item: item[1], reverse=True)[:3])
else:
    print(f"Add a chilli leaf image at {IMAGE_PATH.resolve()} or update IMAGE_PATH.")

## Interactive image upload

Run this cell to open an image upload widget inside Jupyter. This uses a local-only Gradio interface; it does not create a public share link.

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload a chilli leaf image"),
    outputs=gr.Label(num_top_classes=3, label="Top predictions"),
    title="Chilli Leaf Disease Detector",
    description="Experimental image-based screening. Consult an agricultural expert before treatment.",
)
demo.launch(inline=True, share=False)